> **OPTIONAL — nothing downstream reads this notebook's output.** Kiswahili is
> a *source* language in this project, never a target, so the re-translated
> corpus is not used by `03_build_training_data.ipynb`. Run it only if you
> later add an `eng→swh` direction. Skip straight to notebook 03 otherwise.

# 02 · Regenerate Kiswahili targets with NLLB-200 1.3B

**Why this notebook exists.** The Kiswahili column in `psa_parallel_dataset.csv`
was produced by `nllb-200-distilled-600M` — the very model we are about to
fine-tune. Training a model on its own output is self-distillation: it teaches
the model nothing it does not already believe, and any BLEU gain measured
against those targets is circular.

Re-translating with the **1.3B** model makes this genuine sequence-level
knowledge distillation from a stronger teacher into a smaller student, which is
a well-established and defensible technique.

**Inputs** — `output/psa_parallel_dataset.csv`
**Outputs** — `output/psa_en_swh_nllb13b.csv`
**Runtime** — roughly 40–90 minutes on the A100 for ~50k sentences. Resumable.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # so `import nb_common` works
sys.path.insert(0, str(pathlib.Path.cwd()))
import nb_common as C

C.set_seed()
C.use_house_style()
print(f"project root: {C.ROOT}")

In [ ]:
import torch, pandas as pd, math, os, time
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

C.gpu_report()
C.require_files(C.PSA_PARALLEL_CSV)

## 1. Configuration

`BATCH_SIZE` is the knob to turn if you hit out-of-memory. The node is shared —
size against **free** VRAM reported above, not total.

In [ ]:
BATCH_SIZE  = 64      # lower to 32 or 16 if OOM
NUM_BEAMS   = 4       # 1 = greedy and ~3x faster; 4 gives better targets
MAX_NEW     = 128
CHECKPOINT_EVERY = 20  # batches between saves

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if device == "cuda" else torch.float32
print(f"device={device} dtype={dtype} batch={BATCH_SIZE} beams={NUM_BEAMS}")

## 2. Load the teacher

The Kenyan acronyms are added to the tokenizer so they are not shattered into
subwords and mangled. We resize the embedding matrix to match; the new rows are
randomly initialised but these tokens are copied through rather than translated.

In [ ]:
tok = AutoTokenizer.from_pretrained(C.TEACHER_MODEL, src_lang=C.ENG)
added = tok.add_tokens(C.ACRONYMS)
print(f"added {added} acronym tokens")

model = AutoModelForSeq2SeqLM.from_pretrained(C.TEACHER_MODEL, torch_dtype=dtype).to(device)
model.resize_token_embeddings(len(tok))
model.eval()

swh_id = tok.convert_tokens_to_ids(C.SWH)
print(f"forced_bos_token_id for {C.SWH} = {swh_id}")

## 3. Load the source text, resuming if a previous run was interrupted

In [ ]:
psa = pd.read_csv(C.PSA_PARALLEL_CSV)
src_col = "English"
df = psa[["PSA_Id", "Domain", src_col]].copy() if "PSA_Id" in psa.columns \
     else psa[[src_col]].copy()
df["Kiswahili_1p3b"] = ""

if C.PSA_SWH_13B_CSV.exists():
    prev = pd.read_csv(C.PSA_SWH_13B_CSV)
    if len(prev) == len(df):
        df["Kiswahili_1p3b"] = prev["Kiswahili_1p3b"].fillna("").astype(str)
        print(f"resuming: {(df.Kiswahili_1p3b.str.strip() != '').sum():,} already done")

todo = df.index[df.Kiswahili_1p3b.str.strip() == ""].tolist()
print(f"to translate: {len(todo):,} of {len(df):,}")

## 4. Translate

In [ ]:
texts = df[src_col].astype(str).tolist()
n_batches = math.ceil(len(todo) / BATCH_SIZE)
t0 = time.time()

for b in range(n_batches):
    idx = todo[b*BATCH_SIZE:(b+1)*BATCH_SIZE]
    batch = [texts[i] for i in idx]

    enc = tok(batch, return_tensors="pt", padding=True, truncation=True,
              max_length=MAX_NEW).to(device)
    with torch.no_grad():
        out = model.generate(**enc, forced_bos_token_id=swh_id,
                             max_new_tokens=MAX_NEW, num_beams=NUM_BEAMS)
    for i, txt in zip(idx, tok.batch_decode(out, skip_special_tokens=True)):
        df.at[i, "Kiswahili_1p3b"] = txt

    if b % CHECKPOINT_EVERY == 0 or b == n_batches - 1:
        df.to_csv(C.PSA_SWH_13B_CSV, index=False, encoding="utf-8")
        done = (b + 1) * BATCH_SIZE
        rate = done / max(1e-6, time.time() - t0)
        eta = (len(todo) - done) / max(1e-6, rate) / 60
        print(f"batch {b+1}/{n_batches}  {min(done, len(todo)):,}/{len(todo):,}"
              f"  {rate:.1f} sent/s  ETA {eta:.1f} min")

df.to_csv(C.PSA_SWH_13B_CSV, index=False, encoding="utf-8")
print(f"\nwrote {C.PSA_SWH_13B_CSV}")

## 5. Sanity-check the teacher output

Compare the 600M targets you already had against the 1.3B ones. If they were
identical there would be no point to this notebook; if they differ wildly on
short inputs, inspect before trusting them.

In [ ]:
import difflib
merged = psa[[src_col, "Kiswahili"]].join(df[["Kiswahili_1p3b"]])
merged = merged[merged.Kiswahili_1p3b.astype(str).str.strip() != ""]

same = (merged.Kiswahili.astype(str).str.strip() ==
        merged.Kiswahili_1p3b.astype(str).str.strip()).mean()
print(f"identical outputs 600M vs 1.3B: {100*same:.1f}%")

for _, r in merged.sample(min(5, len(merged)), random_state=0).iterrows():
    print("\nEN   :", r[src_col][:100])
    print("600M :", str(r.Kiswahili)[:100])
    print("1.3B :", str(r.Kiswahili_1p3b)[:100])

In [ ]:
del model
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("teacher unloaded. Next: 03_build_training_data.ipynb")